# Phase 6 — Module 3 validation

Held-out datasets vs the two frozen rules. Predictions saved before training; scored after.


In [1]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
import pandas as pd
from virgo import frozen_rules as fr
from experiments import predict_module3 as pm, score_module3 as sm
fr.HELDOUT

/home/m-adam/miniconda/envs/i2v/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['citeseer_linqs', 'proteins', 'pubmed', 'actor', 'minesweeper']

## 1 · Predict — before training


In [2]:
pred, added = pm.freeze_predictions(fr.HELDOUT)
print(f"newly frozen: {added or 'none (already saved before training)'}")
display(
    pred.round(4)
    .rename(columns={
        "homophily_adjusted": "adj_h",
        "rule1_interval": "R1 (adj_h) interval",
        "rule1_pred": "R1 (adj_h) prediction",
        "largest_component_frac": "largest_comp_frac",
        "rule2_interval": "R2 (largest_comp_frac) interval",
        "rule2_pred": "R2 (largest_comp_frac) prediction",
        "predicted_verdict": "final verdict",
    })
    .style
    .hide(axis="index")
    .set_table_styles([
        {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
        {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
        {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
    ])
)

newly frozen: none (already saved before training)


dataset,domain,tasks,adj_h,R1 (adj_h) interval,R1 (adj_h) prediction,largest_comp_frac,R2 (largest_comp_frac) interval,R2 (largest_comp_frac) prediction,final verdict
citeseer_linqs,citation,NC+LP,0.673100,"(0.0926, 0.3613)",keep original,0.646400,"(0.9177, 1.0)",keep original,keep original
proteins,biological,NC+LP,0.355200,"(0.0926, 0.3613)",keep original,0.014300,"(0.9177, 1.0)",keep original,keep original
pubmed,citation,NC+LP,0.686000,"(0.0926, 0.3613)",keep original,1.000000,"(0.9177, 1.0)",augment,rules disagree
actor,film,NC+LP,0.002800,"(0.0926, 0.3613)",augment,1.000000,"(0.9177, 1.0)",augment,augment
minesweeper,grid,NC+LP,0.009400,"(0.0926, 0.3613)",augment,1.000000,"(0.9177, 1.0)",augment,augment


## 2 · Verdict vs actual — after training


In [3]:
scored = sm.score(fr.HELDOUT)
scored.to_csv(sm.SCORED_CSV, index=False)
for r in fr.FROZEN_RULES:
    col = list(scored[f"{r.name}_correct"])
    c = [v for v in col if isinstance(v, bool)]
    skipped = sorted({str(v) for v in col if not isinstance(v, bool)})
    print(f"{r.name} ({r.predictor} {r.op} {r.point}): "
          + (f"{sum(c)}/{len(c)} correct" if c else "nothing scored yet")
          + (f"  [not scored: {', '.join(skipped)}]" if skipped else ""))
display(
    scored.round(4)
    .rename(columns={
        "homophily_adjusted": "adj_h",
        "largest_component_frac": "largest_comp_frac",
        "rule1_pred": "R1 (adj_h) prediction",
        "rule1_correct": "R1 (adj_h) correct",
        "rule2_pred": "R2 (largest_comp_frac) prediction",
        "rule2_correct": "R2 (largest_comp_frac) correct",
        "predicted_verdict": "final verdict",
        "actual_verdict": "actual verdict",
        "best_augmented": "best aug",
        "best_variant": "best variant",
    })
    .style
    .format(na_rep="—")
    .hide(axis="index")
    .set_table_styles([
        {"selector": "table", "props": [("width", "100%"), ("table-layout", "fixed"), ("font-size", "11px")]},
        {"selector": "th", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
        {"selector": "td", "props": [("white-space", "normal"), ("word-wrap", "break-word"), ("text-align", "center"), ("padding", "4px")]},
    ])
)

rule1 (homophily_adjusted < 0.227): 3/4 correct  [not scored: no decision]
rule2 (largest_component_frac > 0.9588): 3/4 correct  [not scored: no decision]


dataset,adj_h,largest_comp_frac,R1 (adj_h) prediction,R1 (adj_h) correct,R2 (largest_comp_frac) prediction,R2 (largest_comp_frac) correct,final verdict,actual verdict,original,best aug,best variant
citeseer_linqs,0.673100,0.646400,keep original,True,keep original,True,keep original,keep original,0.621800,0.543700,centrality
proteins,0.355200,0.014300,keep original,True,keep original,True,keep original,keep original,0.672000,0.583400,centrality
pubmed,0.686000,1.000000,keep original,no decision,augment,no decision,rules disagree,tie,0.639200,0.623800,hybrid
actor,0.002800,1.000000,augment,True,augment,True,augment,augment,0.595300,0.661700,degree
minesweeper,0.009400,1.000000,augment,False,augment,False,augment,keep original,0.709300,0.667100,hybrid
